# TSA Chapter 12: Lag-Window Spectral Estimation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/TSA/blob/main/TSA_ch12/TSA_ch12_lag_windows/TSA_ch12_lag_windows.ipynb)

In [ ]:
!pip install matplotlib numpy scipy statsmodels pywt -q

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Style configuration
COLORS = {
    'blue':   '#1A3A6E',
    'red':    '#DC3545',
    'green':  '#2E7D32',
    'orange': '#E67E22',
    'gray':   '#666666',
    'purple': '#8E44AD',
}

plt.rcParams.update({
    'axes.facecolor':     'none',
    'figure.facecolor':   'none',
    'savefig.transparent': True,
    'axes.spines.top':    False,
    'axes.spines.right':  False,
    'axes.grid':          False,
    'font.size':          9,
    'axes.labelsize':     9,
    'axes.titlesize':     10,
    'legend.fontsize':    8,
    'xtick.labelsize':    8,
    'ytick.labelsize':    8,
})

In [ ]:
# Chart: ch12_lag_windows
# Bartlett, Tukey-Hanning and Parzen lag windows and their spectral windows
M = 64
k = np.arange(-M, M+1)

def bartlett(k, M):     return 1 - np.abs(k)/M
def tukey_hanning(k, M): return 0.5*(1 + np.cos(np.pi*k/M))
def parzen(k, M):
    u = np.abs(k) / M
    return np.where(u <= 0.5, 1 - 6*u**2 + 6*u**3, 2*(1-u)**3)

windows = {
    'Bartlett':      (bartlett(k, M),      COLORS['blue']),
    'Tukey-Hanning': (tukey_hanning(k, M), COLORS['orange']),
    'Parzen':        (parzen(k, M),        COLORS['red']),
}
omega = np.linspace(-np.pi, np.pi, 2048)

fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for name, (w, col) in windows.items():
    axes[0, 0].plot(k, w, color=col, lw=1.8, label=name)
axes[0, 0].set_title('Lag Windows'); axes[0, 0].set_xlabel('Lag k'); axes[0, 0].set_ylabel('w(k)')

for name, (w, col) in windows.items():
    W = np.abs(np.fft.fftshift(np.fft.fft(w, n=2048)))
    axes[0, 1].plot(omega, W / W.max(), color=col, lw=1.8, label=name)
axes[0, 1].set_title('Spectral Windows (normalised)'); axes[0, 1].set_xlim(-np.pi/4, np.pi/4)
axes[0, 1].set_xlabel('Frequency ω'); axes[0, 1].set_ylabel('|W(ω)|')

for name, (w, col) in windows.items():
    W = np.abs(np.fft.fftshift(np.fft.fft(w, n=2048)))
    W = np.maximum(W, 1e-10)
    axes[1, 0].plot(omega, 20*np.log10(W/W.max()), color=col, lw=1.8, label=name)
axes[1, 0].set_title('Spectral Windows (dB)'); axes[1, 0].set_ylim(-80, 5); axes[1, 0].set_xlim(-np.pi/4, np.pi/4)
axes[1, 0].set_xlabel('Frequency ω'); axes[1, 0].set_ylabel('dB')

phi = 0.6; lags = np.arange(-M, M+1)
acvf = phi**np.abs(lags)
for name, (w, col) in windows.items():
    axes[1, 1].plot(lags, acvf * w, color=col, lw=1.8, label=name)
axes[1, 1].set_title('ACVF × Lag Window (AR(1) φ=0.6)'); axes[1, 1].set_xlabel('Lag k'); axes[1, 1].set_ylabel('γ(k)·w(k)')

for ax in axes.flat:
    ax.set_facecolor('none')
    ax.spines[['top','right']].set_visible(False)
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2), ncol=3, frameon=False)
fig.patch.set_alpha(0)
plt.tight_layout()
plt.savefig('ch12_lag_windows.pdf', bbox_inches='tight', dpi=150)
plt.savefig('ch12_lag_windows.png', bbox_inches='tight', dpi=150)
plt.show()